# Lektion 02 - Utforska Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** är ett enhetligt ramverk för att bygga AI-agenter. Det tillhandahåller en ren, sammansättbar arkitektur med fyra kärnkomponenter:

- **Client** – ansluter till en AI-modellendpoint och hanterar kommunikation
- **Agent** – omsluter en klient med instruktioner och verktygsdefinitioner
- **Tools** – utökar agentens kapaciteter med anpassade funktioner som modellen kan anropa
- **Session** – upprätthåller konversationshistorik för flerstegsinteraktioner

I denna lektion bygger vi en **resebokningsagent** som kontrollerar tillgänglighet för destinationer med hjälp av dessa koncept.


## Installation


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Förstå Agent Framework-arkitekturen

Microsoft Agent Framework följer en lagerarkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – En `FoundryChatClient` ansluter till en Azure OpenAI-distribution. Den hanterar autentisering, förfrågningsformat och responsanalys.
2. **Agent** – Skapas från klienten via `provider.create_agent()`, agenten kombinerar modellåtkomst med instruktioner (systemprompt) och verktyg.
3. **Verktyg** – Pythonfunktioner dekorerade med `@tool` som agenten kan anropa för att utföra åtgärder eller hämta data.
4. **Session** – Ett `AgentSession`-objekt (skapat via `agent.create_session()`) som lagrar konversationshistorik, vilket möjliggör fleromgångs-dialoog där agenten kommer ihåg tidigare kontext.

Låt oss bygga varje lager steg för steg.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Lägga till verktyg med @tool-dekoratorn

Verktyg låter agenter utföra handlingar utöver att generera text. `@tool`-dekoratorn omvandlar en vanlig Python-funktion till något som agenten kan anropa.

Viktiga punkter:
- Använd `Annotated[type, "beskrivning"]` så att modellen förstår varje parameter.
- Docstringen blir verktygsbeskrivningen som modellen ser.
- `approval_mode="never_require"` betyder att verktyget körs automatiskt utan användarbekräftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Skapa en agent med verktyg

Nu kombinerar vi klienten, instruktionerna och verktygen till en agent. `instructions` fungerar som systemprompten — de definierar agentens persona och beteende.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Flersamtalssamtal med Sessions

En `AgentSession` (skapad via `agent.create_session()`) håller reda på alla meddelanden i en konversation. Genom att skicka samma session till varje `agent.run()`-anrop har agenten tillgång till hela samtalshistoriken och kan hänvisa tillbaka till tidigare meddelanden.

Vi skickar `tools=[check_destination_availability]` så att agenten kan använda vår tillgänglighetskontroll vid varje varv.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sammanfattning

I den här lektionen undersökte du de fyra pelarna i Microsoft Agent Framework:

| Koncept | Vad du lärde dig |
|---------|------------------|
| **Klient** | `FoundryChatClient` ansluter till Azure OpenAI med autentisering baserad på referenser |
| **Agent** | `provider.create_agent()` paketerar en modellanslutning med instruktioner och ett namn |
| **Verktyg** | `@tool`-dekorationen exponerar Python-funktioner för agenten att kalla på |
| **Session** | `agent.create_session()` bevarar konversationshistorik över flera omgångar |

Dessa byggstenar sammansätts för att skapa agenter som kan ha naturliga konversationer, kalla externa funktioner och bevara kontext — grunden för mer avancerade agentmönster i senare lektioner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
